# Batch Inference: Summarizing Reviews at Scale

So far we've called models one request at a time (`invoke_model`, `converse`). **Batch inference** is a different tool for a different job: running a model over a large volume of inputs where you don't need an instant answer.

How it works:
1. Write a **JSONL manifest** - one line per request, each with a `recordId` and a `modelInput` (the same body you'd pass to `invoke_model`).
2. Upload it to **Amazon S3**.
3. Submit an **async job**. Bedrock processes every record and writes results to an S3 output location.

> **Teaching/Learning Tip:** Batch is the opposite tradeoff from streaming. Streaming optimizes for *low latency* - one answer, right now. Batch optimizes for *cost and throughput* on huge volumes, and in exchange you accept that it's not instant (jobs queue and can take minutes to hours). There's also a per-model **minimum record count** (a service quota), so batch is deliberately not for one-off calls.

## Build one record

Each manifest line has a `recordId` (your label to match inputs to outputs) and a `modelInput` (exactly the body shape the model expects). This matches the slide.

In [ ]:
import json

MODEL_ID = "amazon.nova-lite-v1:0"
SYSTEM_PROMPT = (
    "You are a travel expert AI assistant. Create comprehensive, engaging city "
    "summaries based on user reviews."
)


def build_record(record_id, city):
    prompt = (
        f"Use the following reviews for {city} {{{{ REVIEWS EXCLUDED }}}}\n\n"
        f"Please provide a well-structured summary that includes:\n"
        f"1. Overall impression and sentiment\n2. Key highlights"
    )
    return {
        "recordId": record_id,
        "modelInput": {
            "schemaVersion": "messages-v1",
            "messages": [{"role": "user", "content": [{"text": prompt}]}],
            "system": [{"text": SYSTEM_PROMPT}],
            "inferenceConfig": {
                "maxTokens": 500, "topP": 0.9, "topK": 20, "temperature": 0.7,
            },
        },
    }


print(json.dumps(build_record("albuquerque-0000", "Albuquerque"), indent=2))

## Build the full manifest

We repeat a base list of cities to reach 1000 records - safely above the per-model minimum. In a real pipeline these would be distinct records from your data (e.g. one per city, each with its actual reviews).

> **Teaching/Learning Tip:** The exact minimum is a *model-specific quota* - check the Bedrock console (Service Quotas) for "Minimum number of records per batch inference job" for your model. It's high enough that you can't hand-write a manifest, which is the whole point of batch: bulk work.

In [ ]:
BASE_CITIES = [
    "Albuquerque", "Denver", "Seattle", "Austin", "Miami", "Chicago",
    "Portland", "Nashville", "Boston", "Phoenix",
]
RECORD_COUNT = 1000
MANIFEST_FILE = "batch_manifest.jsonl"

with open(MANIFEST_FILE, "w") as f:
    for i in range(RECORD_COUNT):
        city = BASE_CITIES[i % len(BASE_CITIES)]
        f.write(json.dumps(build_record(f"{city.lower()}-{i:04d}", city)) + "\n")

line_count = sum(1 for _ in open(MANIFEST_FILE))
print(f"Wrote {line_count} records to {MANIFEST_FILE}")

## Submit the batch job (guarded)

This is the real batch API. It's **guarded** - fill in your bucket and role ARN and uncomment to run it. It needs:
- an **S3 bucket** you can write to (for input and output)
- an **IAM service role** whose trust policy lets `bedrock.amazonaws.com` assume it, with S3 read/write on those prefixes

> **Teaching/Learning Tip:** Because batch is async, submitting returns immediately with a job ARN - the work happens later. In a multi-day class you can submit now and check the results tomorrow. Don't wait on it live.

In [ ]:
import time
import boto3

REGION = "us-east-1"

# --- Fill these in, then uncomment the block below to actually submit ---
BUCKET = "your-bucket-name"
ROLE_ARN = "arn:aws:iam::<account-id>:role/<bedrock-batch-role>"

# s3 = boto3.client("s3", region_name=REGION)
# bedrock = boto3.client("bedrock", region_name=REGION)
#
# input_key = f"batch-input/{MANIFEST_FILE}"
# s3.upload_file(MANIFEST_FILE, BUCKET, input_key)
#
# response = bedrock.create_model_invocation_job(
#     jobName=f"summarize-reviews-{int(time.time())}",
#     roleArn=ROLE_ARN,
#     modelId=MODEL_ID,
#     inputDataConfig={"s3InputDataConfig": {"s3Uri": f"s3://{BUCKET}/{input_key}"}},
#     outputDataConfig={"s3OutputDataConfig": {"s3Uri": f"s3://{BUCKET}/batch-output/"}},
# )
# job_arn = response["jobArn"]
# print("Submitted:", job_arn)
print("Submission cell is guarded - fill in BUCKET/ROLE_ARN and uncomment to run.")

## Check job status (guarded)

Once submitted, poll the job with its ARN. The progress counters tell you how many of the total records have been processed - no need to peek at S3.

In [ ]:
# bedrock = boto3.client("bedrock", region_name=REGION)
# job = bedrock.get_model_invocation_job(jobIdentifier=job_arn)
# print("Status:", job["status"])
# print("Output:", job["outputDataConfig"]["s3OutputDataConfig"]["s3Uri"])
print("Status cell is guarded - uncomment after submitting a job.")